In [1]:
import os
import sys
import numpy as np
import scipy.io as scio
from PIL import Image

import torch
# from torch._six import container_abcs
from torch.utils.data import Dataset
from tqdm import tqdm

In [2]:
root = 'F:\graspnet'
camera = 'kinect'
load_label = True
remove_outlier = False
num_points = 10000

In [3]:
sceneIds = list(range(10))
sceneIds = ['scene_{}'.format(str(x).zfill(4)) for x in sceneIds]

In [4]:
colorpath, depthpath, labelpath, metapath, scenename, frameid = [], [], [], [], [], []
collision_labels = {}

In [44]:
class CameraInfo():
    """ Camera intrisics for point cloud creation. """
    def __init__(self, width, height, fx, fy, cx, cy, scale):
        self.width = width
        self.height = height
        self.fx = fx
        self.fy = fy
        self.cx = cx
        self.cy = cy
        self.scale = scale

def create_point_cloud_from_depth_image(depth, camera, organized=True):
    """ Generate point cloud using depth image only.

        Input:
            depth: [numpy.ndarray, (H,W), numpy.float32]
                depth image
            camera: [CameraInfo]
                camera intrinsics
            organized: bool
                whether to keep the cloud in image shape (H,W,3)

        Output:
            cloud: [numpy.ndarray, (H,W,3)/(H*W,3), numpy.float32]
                generated cloud, (H,W,3) for organized=True, (H*W,3) for organized=False
    """
    assert(depth.shape[0] == camera.height and depth.shape[1] == camera.width)
    xmap = np.arange(camera.width)
    ymap = np.arange(camera.height)
    xmap, ymap = np.meshgrid(xmap, ymap)
    points_z = depth / camera.scale
    points_x = (xmap - camera.cx) * points_z / camera.fx
    points_y = (ymap - camera.cy) * points_z / camera.fy
    cloud = np.stack([points_x, points_y, points_z], axis=-1)
    if not organized:
        cloud = cloud.reshape([-1, 3])
    return cloud

def get_workspace_mask(cloud, seg, trans=None, organized=True, outlier=0):
    """ Keep points in workspace as input.

        Input:
            cloud: [np.ndarray, (H,W,3), np.float32]
                scene point cloud
            seg: [np.ndarray, (H,W,), np.uint8]
                segmantation label of scene points
            trans: [np.ndarray, (4,4), np.float32]
                transformation matrix for scene points, default: None.
            organized: [bool]
                whether to keep the cloud in image shape (H,W,3)
            outlier: [float]
                if the distance between a point and workspace is greater than outlier, the point will be removed
                
        Output:
            workspace_mask: [np.ndarray, (H,W)/(H*W,), np.bool]
                mask to indicate whether scene points are in workspace
    """
    if organized:
        h, w, _ = cloud.shape
        cloud = cloud.reshape([h*w, 3])
        seg = seg.reshape(h*w)
    if trans is not None:
        cloud = transform_point_cloud(cloud, trans)
    foreground = cloud[seg>0]
    xmin, ymin, zmin = foreground.min(axis=0)
    xmax, ymax, zmax = foreground.max(axis=0)
    mask_x = ((cloud[:,0] > xmin-outlier) & (cloud[:,0] < xmax+outlier))
    mask_y = ((cloud[:,1] > ymin-outlier) & (cloud[:,1] < ymax+outlier))
    mask_z = ((cloud[:,2] > zmin-outlier) & (cloud[:,2] < zmax+outlier))
    workspace_mask = (mask_x & mask_y & mask_z)
    if organized:
        workspace_mask = workspace_mask.reshape([h, w])

    return workspace_mask

def load_grasp_labels(root):
    obj_names = list(range(88))
    valid_obj_idxs = []
    grasp_labels = {}
    for i, obj_name in enumerate(tqdm(obj_names, desc='Loading grasping labels...')):
        if i == 18: continue
        valid_obj_idxs.append(i + 1) #here align with label png
        label = np.load(os.path.join(root, 'grasp_label', '{}_labels.npz'.format(str(i).zfill(3))))
        tolerance = np.load(os.path.join(root, 'tolerance', '{}_tolerance.npy'.format(str(i).zfill(3))))
        grasp_labels[i + 1] = (label['points'].astype(np.float32), label['offsets'].astype(np.float32),
                                label['scores'].astype(np.float32), tolerance)

    return valid_obj_idxs, grasp_labels

In [45]:
valid_obj_idxs, grasp_labels = load_grasp_labels(root)

Loading grasping labels...: 100%|██████████| 88/88 [01:39<00:00,  1.13s/it]


In [6]:
for x in sceneIds:
    for img_num in range(10):
        colorpath.append(os.path.join(root, 'scenes', x, camera, 'rgb', str(img_num).zfill(4)+'.png'))
        depthpath.append(os.path.join(root, 'scenes', x, camera, 'depth', str(img_num).zfill(4)+'.png'))
        labelpath.append(os.path.join(root, 'scenes', x, camera, 'label', str(img_num).zfill(4)+'.png'))
        metapath.append(os.path.join(root, 'scenes', x, camera, 'meta', str(img_num).zfill(4)+'.mat'))
        scenename.append(x.strip())
        frameid.append(img_num)
    if load_label:
        collision_labels_temp = np.load(os.path.join(root, 'collision_label', x.strip(),  'collision_labels.npz'))
        collision_labels[x.strip()] = {}
        for i in range(len(collision_labels_temp)):
            collision_labels[x.strip()][i] = collision_labels_temp['arr_{}'.format(i)]

In [10]:
# def get_data_label(index):
index = 0
color = np.array(Image.open(colorpath[index]), dtype=np.float32)/ 255.0
depth = np.array(Image.open(depthpath[index])) #(720, 1280)
seg = np.array(Image.open(labelpath[index]))
meta = scio.loadmat(metapath[index])
scene = scenename[index]
obj_idxs = meta['cls_indexes'].flatten().astype(np.int32)
poses = meta['poses']
intrinsic = meta['intrinsic_matrix']
factor_depth = meta['factor_depth']
camera_t = CameraInfo(1280.0, 720.0, intrinsic[0][0], intrinsic[1][1], intrinsic[0][2], intrinsic[1][2], factor_depth)
cloud = create_point_cloud_from_depth_image(depth, camera_t, organized=True) #(720, 1280, 3)

# get valid points
depth_mask = (depth > 0)
seg_mask = (seg > 0)
if remove_outlier:
    camera_poses = np.load(os.path.join(root, 'scenes', scene, camera, 'camera_poses.npy'))
    align_mat = np.load(os.path.join(root, 'scenes', scene, camera, 'cam0_wrt_table.npy'))
    trans = np.dot(align_mat, camera_poses[frameid[index]])
    workspace_mask = get_workspace_mask(cloud, seg, trans=trans, organized=True, outlier=0.02)
    mask = (depth_mask & workspace_mask)
else:
    mask = depth_mask
cloud_masked = cloud[mask]
color_masked = color[mask]
seg_masked = seg[mask]
# if return_raw_cloud:
#     return cloud_masked, color_masked

In [71]:
# temp_obj_list = []
# for sce in os.listdir(f'{root}/scenes'):
#     obj_file = f'{root}/scenes/{sce}/object_id_list.txt'
#     with open(obj_file, "r", encoding="utf-8") as file:
#         for line in file:
#             obj_index = int(line.strip())
#             temp_obj_list.append(obj_index)
# temp_obj_set = set(temp_obj_list)

In [39]:
if len(cloud_masked) >= num_points:
    idxs = np.random.choice(len(cloud_masked), num_points, replace=False)
else:
    idxs1 = np.arrange(len(cloud_masked))
    idxs2 = np.random(len(cloud_masked), num_points-len(cloud_masked), replace=True)
    idxs = np.concatenate([idxs1, idxs2], axis=0)
cloud_sampled = cloud_masked[idxs]
color_sampled = color_masked[idxs]
seg_sampled = seg_masked[idxs]
objectness_label = seg_sampled.copy()
objectness_label[objectness_label>1] = 1


In [ ]:
object_poses_list = []
grasp_points_list = []
grasp_offsets_list = []
grasp_scores_list = []
grasp_tolerance_list = []

for i, obj_idx in enumerate(obj_idxs):
    print(type(obj_idx))
    if obj_idx not in valid_obj_idxs:
        continue
    if (seg_sampled == obj_idx).sum() < 50:
        continue
    object_poses_list.append(poses[:, :, i])
    points, offsets, scores, tolerances = grasp_labels[obj_idx]
    collision = collision_labels[scene][i]

    idxs = np.random.choice(len(points), min(max(int(len(points)/4),300),len(points)), replace=False)
    grasp_points_list.append(points[idxs])
    grasp_offsets_list.append(offsets[idxs])
    collision = collision[idxs].copy()
    scores = scores[idxs].copy()
    scores[collision] = 0
    grasp_scores_list.append(scores)
    

In [70]:
grasp_labels

{1: (array([[-0.0122905 ,  0.07282408,  0.10049217],
         [ 0.02331672,  0.07790881,  0.01449282],
         [ 0.027253  ,  0.07622192,  0.09785532],
         ...,
         [ 0.031894  ,  0.01771812,  0.07439509],
         [-0.02498909, -0.05372961, -0.11060694],
         [ 0.01005789,  0.07523085,  0.07048508]],
        shape=(3459, 3), dtype=float32),
  array([[[[[0.        , 0.01      , 0.        ],
            [0.        , 0.02      , 0.        ],
            [0.        , 0.03      , 0.        ],
            [0.        , 0.04      , 0.        ]],
  
           [[0.2617994 , 0.01      , 0.        ],
            [0.2617994 , 0.02      , 0.        ],
            [0.2617994 , 0.03      , 0.        ],
            [0.2617994 , 0.04      , 0.        ]],
  
           [[0.5235988 , 0.01      , 0.        ],
            [0.5235988 , 0.02      , 0.        ],
            [0.5235988 , 0.03      , 0.        ],
            [0.5235988 , 0.04      , 0.        ]],
  
           ...,
  
          

In [122]:
poses.shape

(3, 4, 9)

In [95]:
cloud_masked.shape

(846916, 3)

In [85]:
depth.shape

(720, 1280)